# 23 · When do predictive representations help decisions?

This lesson runs the matched visual-JEPA decision study. It uses the same small synthetic MountainCar task as Notebook 22, but it locks the planner on development starts before it opens held-out cases.

**Status:** the study is reproducible from the retained source and evidence. It is a local synthetic-task experiment. It is not a reproduction of V-JEPA 2, LeJEPA, VISReg, or a robotics result.


## Research question

> When do predictive representations actually improve action selection, and when does accurate prediction fail to translate into effective decisions?

The experiment separates one-step prediction, changed-action effects, probe quality, planner outcomes, and compute. It compares exact dynamics, learned state dynamics, image-reconstruction dynamics, action-conditioned visual JEPA dynamics, and a direct visual-momentum rule. The direct rule is hand-written. The state and oracle models have privileged state access.


## Run order and stop gates

1. **CPU smoke:** collect reserved smoke data, train small versions of all three model families, and run short planner cases. This does not open final evaluation seeds.
2. **Development search:** tune exact-dynamics CEM on seeds 8200–8209 and confirm once on 8210–8219. The run records the complete planner configuration and hashes the preregistration and source.
3. **Held-out oracle gate:** test the locked exact planner on seeds 8700–8719. The learned-model stage opens only after at least 18 of 20 successes.
4. **Full comparison:** train five model seeds per family, then evaluate all methods on the shared starts. The run uses CUDA when available.

If a gate fails, keep its records and stop at that stage. Do not adjust the gate after reading the results.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "scripts/run_visual_jepa_decision_study.py").is_file():
    root = root.parent
script = root / "scripts/run_visual_jepa_decision_study.py"
output_dir = Path(os.environ.get(
    "JEPA_RUN_OUTPUT_DIR",
    "/mnt/e/codex-research/jepa-world-models-tutorial/visual-jepa-decision-study-v1",
))
print(f"Repository: {root}")
print(f"Heavy run output: {output_dir}")


In [ ]:
# CPU smoke only. It does not read or write final evaluation cases.
smoke_dir = output_dir / "notebook-smoke"
subprocess.run(
    [sys.executable, str(script), "smoke", "--output-dir", str(smoke_dir)],
    cwd=root,
    check=True,
)


## Optional full run

Use the E:-hosted CUDA environment and a run directory on E: or D:. The v1 run takes hours. Keep its frozen protocol and row counts.

```bash
export UV_PROJECT_ENVIRONMENT=/mnt/e/codex-research/jepa-world-models-tutorial/.venv
export JEPA_RUN_OUTPUT_DIR=/mnt/e/codex-research/jepa-world-models-tutorial/visual-jepa-decision-study-v1
uv run --locked --no-sync python scripts/run_visual_jepa_decision_study.py dev-search --output-dir "$JEPA_RUN_OUTPUT_DIR" --device cuda
uv run --locked --no-sync python scripts/run_visual_jepa_decision_study.py oracle-gate --output-dir "$JEPA_RUN_OUTPUT_DIR" --device cpu
uv run --locked --no-sync python scripts/run_visual_jepa_decision_study.py full --output-dir "$JEPA_RUN_OUTPUT_DIR" --device cuda
uv run --locked --no-sync python scripts/run_visual_jepa_decision_study.py validate --output-dir "$JEPA_RUN_OUTPUT_DIR"
uv run --locked --no-sync python scripts/analyze_visual_jepa_decision_study.py
uv run --locked --no-sync python scripts/generate_visual_jepa_decision_study_figures.py
```

Set `UV_PROJECT_ENVIRONMENT` to the prepared E: environment only when its CUDA runtime has been verified. Keep the CPU environment for CPU-only checks.

## Inspect retained evidence

The runner copies compact CSV and JSON records into `docs/evidence/`. It stores model checkpoints only in the E: run directory. The built-in run report is retained as execution evidence; the separate analysis JSON contains the shared-start bootstrap estimate. The run may have complete records while its scientific comparison is still partial because of the JEPA spread warning.

The figure command rebuilds one A–D results dashboard and one E architecture diagram. It does not train models or change the report.

In [ ]:
analysis_path = root / "docs/evidence/visual-jepa-decision-study-v1-analysis.json"
if analysis_path.is_file():
    import json
    analysis = json.loads(analysis_path.read_text(encoding="utf-8"))
    print("Record rows:", analysis.get("completed_episode_rows"))
    print("Claim status:", analysis.get("claim_status"))
    print("Reason:", analysis.get("claim_status_reason"))
    print("Primary comparison:", analysis.get("primary_comparison"))
else:
    print("No complete evidence set is available yet. Complete and validate the gated study first.")

## Claim limits

This experiment uses a hand-rendered task, small models trained from scratch, and one synthetic control environment. The visual dynamics depend on a training-only/validation-selected linear state probe. The intervention continuation score uses simulator state as an evaluator. These choices support a bounded local comparison. They do not show that learned JEPA models generally improve planning, that the methods transfer to robots, or that a published paper has been reproduced.

The larger project roadmap still calls for a real learned-JEPA-versus-direct-policy comparison on public visual-control data, with policy-guided and unconstrained search as separate baselines.
